# 10 · Pangolin — an independent splice model

**Pangolin** (Zeng & Li 2022, *Genome Biology* 23:103, PMID 35449021,
[github.com/tkzeng/Pangolin](https://github.com/tkzeng/Pangolin)) is a deep-learning
splice predictor trained on splice-site *usage* across tissues and species. It gives
**one 0-1 score** per variant (no four deltas to juggle) and uses the **same** 0.5 / 0.2
thresholds as SpliceAI. Because the two were built independently, **agreement between
SpliceAI and Pangolin** is stronger evidence than either alone.

> ✅ **Now runnable for REAL.** Pangolin has no precomputed release and is not in dbNSFP,
> but **`build_pangolin.py` runs the actual model locally** — weights ship inside the pip
> package, and it needs only the ~215 kb CFTR reference region (no whole-genome download).
> The curated run below is **genuine Pangolin output** on the classic CF splice alleles
> (with **correct CFTR2 coordinates**), but is labelled `source='DEMO'` because its
> *coverage* is a handful of variants, not a real-scale worklist. It is validated against
> real SpliceAI (e.g. c.2988+1G>A: Pangolin 0.86 vs SpliceAI 0.99).

In [1]:
import sys, pathlib
# `toolkit` is THIS repo's toolkit.py (one directory up) — NOT a pip
# package and nothing to do with gnomAD. The line below puts the repo
# root on sys.path so `import toolkit` resolves to ../toolkit.py.
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
# %matplotlib inline is a Jupyter magic: it draws matplotlib plots inline below the cell
%matplotlib inline

## 1 · Pangolin — a newer splice model, one tidy score

**Pangolin** (Zeng & Li 2022, *Genome Biology* 23:103, PMID **35449021**) was trained on
splice-site *usage* measured across multiple tissues and species, and tends to be a
little more sensitive than SpliceAI for some variant classes. For our purposes the nice
thing is that it returns a single 0-1 score (larger of the biggest splice-usage gain and
loss), directly comparable to SpliceAI's DS_max at the same 0.5 / 0.2 cut-points.

## 2 · Load the demo splice table and tier by Pangolin score

The demo splice frame carries a `pangolin_score`; tier it with the published thresholds.

In [2]:
splice = tk.load_splice_demo()
splice = splice[splice['legacy_name'] != 'syn context'].reset_index(drop=True)  # drop synthetic placeholder
HIGH = tk.THRESHOLDS['pangolin']['high']       # 0.5
MOD  = tk.THRESHOLDS['pangolin']['moderate']   # 0.2
def pangolin_tier(s):
    return 'HIGH' if s >= HIGH else ('MODERATE' if s >= MOD else 'LOW')
splice['pangolin_tier'] = splice['pangolin_score'].apply(pangolin_tier)
splice[['legacy_name', 'variant_type', 'pangolin_score', 'pangolin_tier', 'cftr2_class', 'source']]

,legacy_name,variant_type,pangolin_score,pangolin_tier,cftr2_class,source
0,3849+10kb C>T,deep_intronic,0.79,HIGH,CF-causing,DEMO
1,2789+5G>A,splice_site,0.88,HIGH,CF-causing,DEMO
2,3272-26A>G,deep_intronic,0.73,HIGH,CF-causing,DEMO
3,2657+3A>G,splice_site,0.85,HIGH,CF-causing,DEMO
4,IVS8_5T,deep_intronic,0.22,MODERATE,VUS,DEMO
5,2988+1G>A,splice_site,0.96,HIGH,CF-causing,DEMO
6,1811+1.6kbA>G,deep_intronic,0.65,HIGH,CF-causing,DEMO
7,c.2657+120C>T,deep_intronic,0.51,HIGH,VUS,DEMO


## 3 · REAL Pangolin scores — `build_pangolin.py`

Unlike SpliceAI there is no precomputed file to download; you **run the model**. It is
lighter than it sounds — no GPU or whole-genome FASTA required:

```bash
pip install "git+https://github.com/tkzeng/Pangolin.git" pyfaidx gffutils
python build_pangolin.py     # -> data/pangolin_cftr.csv
```

`build_pangolin.py` loads the bundled weights, pulls the ~215 kb CFTR region from Ensembl
(cached), and scores each variant on a ±5000 bp window using **authoritative CFTR2
coordinates** (not hand-entered ones). `tk.load_pangolin()` then returns the result. A
small curated run stays `source='DEMO'` by scope; run it over a real target set to promote
Pangolin to REAL.

## Example: the shared splice worked-example panel, scored by **Pangolin**

The same fixed panel of famous CFTR **splice** variants runs through every splice tool
(notebooks 09–11), so you can follow one set of variants across the series. The
variant list is `tk.A1_PANEL_VARIANTS` / `tk.A2_KNOWN_CDNA` (shared in `toolkit.py`); the
**scoring is shown inline below** so you can see exactly how Pangolin is joined onto it.

> The assembled cross-tool table for all tools at once is `tk.a2_panel(cadd=True)`.

In [3]:
# Pangolin scores come from RUNNING the model (build_pangolin.py) on the same known
# CF splice alleles with correct CFTR2 coords. tk.load_pangolin() reads that output.
pg = tk.load_pangolin()
pg[['cdna_name', 'legacy_name', 'pangolin_score', 'source']].reset_index(drop=True)

,cdna_name,legacy_name,pangolin_score,source
0,c.3718-2477C>T,3849+10kbC->T,0.3327,DEMO
1,c.2657+5G>A,2789+5G->A,0.8194,DEMO
2,c.3140-26A>G,3272-26A->G,0.8120,DEMO
3,c.2988+1G>A,3120+1G->A,0.8568,DEMO
4,c.1680-886A>G,1811+1634A->G,0.7057,DEMO


## Key takeaways

1. **Pangolin** gives one 0-1 score with the **same** 0.5 / 0.2 thresholds as SpliceAI.
2. SpliceAI + Pangolin **agreeing** is stronger evidence than either alone — and here they
   do (Pangolin recovers HIGH on the canonical CF splice alleles, matching real SpliceAI).
3. `build_pangolin.py` produces **real** Pangolin scores locally; a curated run is labelled
   DEMO by scope. Correct citation: **Zeng & Li 2022, PMID 35449021**.

**Next:** notebook 11 — **CADD**, a real, live score.